# 01_v2 Data Overview And Audit

Stage 01 audits only the active v2 raw data in `_data/01_raw`. It does not preprocess, clean, join into modeling data, feature-engineer, train models, edit raw files, or write to `_data/02_interim`.


In [1]:
import csv
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from statistics import mean, median

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "_data" / "01_raw"
TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "01_v2_data_overview_and_audit"
DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "01_v2_data_overview_and_audit"

RAW_FILES = {
    "Membership": RAW_DIR / "Membership.csv",
    "User_Mapping": RAW_DIR / "User_Mapping.csv",
    "View_History": RAW_DIR / "View_History.csv",
    "Movie_Master": RAW_DIR / "Movie_Master.csv",
}
REQUIRED_CSVS = [
    "01_v2_raw_file_inventory.csv",
    "01_v2_schema_summary.csv",
    "01_v2_membership_target_summary.csv",
    "01_v2_membership_duplicate_audit.csv",
    "01_v2_membership_target_conflict_rows.csv",
    "01_v2_membership_duration_distribution.csv",
    "01_v2_membership_value_anomaly_summary.csv",
    "01_v2_usermapping_cardinality_audit.csv",
    "01_v2_viewhistory_basic_audit.csv",
    "01_v2_viewhistory_duplicate_audit.csv",
    "01_v2_membership_view_temporal_audit.csv",
    "01_v2_moviemaster_duplicate_audit.csv",
    "01_v2_audit_final_checks.csv",
]
MARKDOWN_REPORT = "01_v2_data_audit_report.md"
TARGET_COL = "is_repurchase"
CORE_EVENT_FIELDS = ["USER_KEY", "product_code", "price", "max_screen", "reg_date", "end_date", "payment_device", "billing_method"]

TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

raw_before = {name: {"mtime_ns": path.stat().st_mtime_ns, "size": path.stat().st_size} for name, path in RAW_FILES.items() if path.exists()}

def rel(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")

def read_csv_rows(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        return reader.fieldnames or [], rows

def write_csv(path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fieldnames})

def write_json(path, payload):
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def parse_date_yy(value):
    return datetime.strptime(value, "%y-%m-%d").date()

def parse_date_yyyymmdd(value):
    return datetime.strptime(value, "%Y%m%d").date()

def safe_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def infer_dtype(values):
    non_missing = [v for v in values if v != ""]
    if not non_missing:
        return "empty"
    checks = {
        "integer_like": lambda v: safe_int(v) is not None,
        "float_like": lambda v: safe_float(v) is not None,
        "date_yy_mm_dd": lambda v: _can_parse(v, "%y-%m-%d"),
        "date_yyyymmdd": lambda v: _can_parse(v, "%Y%m%d"),
    }
    for dtype, fn in checks.items():
        if all(fn(v) for v in non_missing):
            return dtype
    return "string_or_mixed"

def _can_parse(value, fmt):
    try:
        datetime.strptime(value, fmt)
        return True
    except ValueError:
        return False

def percentile(sorted_values, p):
    if not sorted_values:
        return ""
    if len(sorted_values) == 1:
        return sorted_values[0]
    index = (len(sorted_values) - 1) * p
    lo = int(index)
    hi = min(lo + 1, len(sorted_values) - 1)
    frac = index - lo
    return sorted_values[lo] * (1 - frac) + sorted_values[hi] * frac

def add_count_rows(rows, section, metric, counter, total=None):
    denominator = total if total is not None else sum(counter.values())
    for value, count in counter.most_common():
        rows.append({
            "section": section,
            "metric": metric,
            "value": value,
            "count": count,
            "rate": round(count / denominator, 6) if denominator else 0,
            "note": "",
        })

loaded = {}
for name, path in RAW_FILES.items():
    columns, rows = read_csv_rows(path)
    loaded[name] = {"path": path, "columns": columns, "rows": rows}

membership = loaded["Membership"]["rows"]
mapping = loaded["User_Mapping"]["rows"]
views = loaded["View_History"]["rows"]
movies = loaded["Movie_Master"]["rows"]

# 1. Raw file inventory and schema summary
raw_file_inventory = []
schema_summary = []
for name, info in loaded.items():
    rows = info["rows"]
    cols = info["columns"]
    total_missing = 0
    dtype_pairs = []
    missing_pairs = []
    for col in cols:
        values = [row.get(col, "") for row in rows]
        missing = sum(1 for v in values if v == "")
        total_missing += missing
        dtype = infer_dtype(values)
        dtype_pairs.append(f"{col}:{dtype}")
        missing_pairs.append(f"{col}:{missing}")
        non_missing = [v for v in values if v != ""]
        schema_summary.append({
            "dataset": name,
            "column_name": col,
            "inferred_dtype": dtype,
            "row_count": len(rows),
            "missing_count": missing,
            "missing_rate": round(missing / len(rows), 6) if rows else 0,
            "unique_count": len(set(values)),
            "unique_count_excluding_missing": len(set(non_missing)),
            "min_value": min(non_missing) if non_missing else "",
            "max_value": max(non_missing) if non_missing else "",
            "top_values": "|".join([f"{v}:{c}" for v, c in Counter(values).most_common(10)]),
        })
    raw_file_inventory.append({
        "dataset": name,
        "file_name": info["path"].name,
        "relative_path": rel(info["path"]),
        "sheet_name_if_applicable": "N/A_csv",
        "row_count": len(rows),
        "column_count": len(cols),
        "column_names": "|".join(cols),
        "inferred_dtypes": "|".join(dtype_pairs),
        "missing_value_counts": "|".join(missing_pairs),
        "total_missing_values": total_missing,
        "file_size_bytes": info["path"].stat().st_size,
    })

# 2. Membership audit
membership_cols = loaded["Membership"]["columns"]
target_counts = Counter(row[TARGET_COL] for row in membership)
user_key_counts = Counter(row["USER_KEY"] for row in membership)
full_row_counts = Counter(tuple(row.get(c, "") for c in membership_cols) for row in membership)
non_target_cols = [c for c in membership_cols if c != TARGET_COL]
non_target_counts = Counter(tuple(row.get(c, "") for c in non_target_cols) for row in membership)
strict_groups = defaultdict(list)
core_groups = defaultdict(list)
for row_number, row in enumerate(membership, start=2):
    strict_groups[tuple(row.get(c, "") for c in non_target_cols)].append((row_number, row))
    core_groups[tuple(row.get(c, "") for c in CORE_EVENT_FIELDS)].append((row_number, row))
strict_conflict_groups = {k: g for k, g in strict_groups.items() if len({row[TARGET_COL] for _, row in g}) > 1}
core_conflict_groups = {}
secondary_fields = [c for c in membership_cols if c not in CORE_EVENT_FIELDS + [TARGET_COL]]
for key, group in core_groups.items():
    if len(group) <= 1:
        continue
    targets = {row[TARGET_COL] for _, row in group}
    secondary_signatures = {tuple(row.get(c, "") for c in secondary_fields) for _, row in group}
    if len(targets) > 1 or len(secondary_signatures) > 1:
        core_conflict_groups[key] = group

membership_target_summary = []
membership_target_summary.append({"section": "target", "metric": "total_row_count", "value": "all", "count": len(membership), "rate": 1, "note": "Membership row is interpreted as subscription event."})
add_count_rows(membership_target_summary, "target", "is_repurchase_distribution", target_counts, len(membership))
membership_target_summary.append({"section": "USER_KEY", "metric": "USER_KEY_cardinality", "value": "unique_USER_KEY", "count": len(user_key_counts), "rate": round(len(user_key_counts)/len(membership), 6), "note": "Same USER_KEY alone is not target conflict."})
membership_target_summary.append({"section": "USER_KEY", "metric": "duplicate_USER_KEY_count", "value": "keys_with_multiple_rows", "count": sum(1 for c in user_key_counts.values() if c > 1), "rate": "", "note": "May represent multiple subscription events."})
membership_target_summary.append({"section": "USER_KEY", "metric": "duplicate_USER_KEY_extra_rows", "value": "extra_rows", "count": sum(c - 1 for c in user_key_counts.values() if c > 1), "rate": "", "note": "Not treated as conflict by itself."})
for field in ["price", "is_promotion", "max_screen", "age", "gender", "is_user_verified", "is_churn_prevented"]:
    add_count_rows(membership_target_summary, field, f"{field}_distribution", Counter(row.get(field, "") for row in membership), len(membership))
price_promo_counter = Counter((row.get("price", ""), row.get("is_promotion", "")) for row in membership)
for (price, promo), count in price_promo_counter.most_common():
    membership_target_summary.append({"section": "price_vs_is_promotion", "metric": "price_is_promotion_cross_tab", "value": f"price={price}|is_promotion={promo}", "count": count, "rate": round(count/len(membership), 6), "note": "Audit only. No rule applied."})

membership_duplicate_audit = [
    {"audit_item": "exact_duplicate_rows", "definition": "all Membership columns identical", "group_count": sum(1 for c in full_row_counts.values() if c > 1), "row_count": sum(c for c in full_row_counts.values() if c > 1), "extra_row_count": sum(c - 1 for c in full_row_counts.values() if c > 1), "recommendation": "Audit before any deduplication."},
    {"audit_item": "duplicate_rows_excluding_is_repurchase", "definition": "all non-target fields identical", "group_count": sum(1 for c in non_target_counts.values() if c > 1), "row_count": sum(c for c in non_target_counts.values() if c > 1), "extra_row_count": sum(c - 1 for c in non_target_counts.values() if c > 1), "recommendation": "Only rows with conflicting target require special label-policy review."},
    {"audit_item": "target_conflict_groups_strict", "definition": "all non-target fields identical but is_repurchase differs", "group_count": len(strict_conflict_groups), "row_count": sum(len(g) for g in strict_conflict_groups.values()), "extra_row_count": "", "recommendation": "Do not choose Y or N for performance. Exclude or flag until business rule is documented."},
    {"audit_item": "core_event_conflict_groups", "definition": "core subscription fields identical, target or secondary attributes differ", "group_count": len(core_conflict_groups), "row_count": sum(len(g) for g in core_conflict_groups.values()), "extra_row_count": "", "recommendation": "Review as ambiguous subscription-event candidates."},
]
target_conflict_rows = []
for conflict_id, (key, group) in enumerate(strict_conflict_groups.items(), start=1):
    targets = "|".join(sorted({row[TARGET_COL] for _, row in group}))
    for row_number, row in group:
        out = {"conflict_level": "strict", "conflict_id": conflict_id, "source_row_number": row_number, "group_size": len(group), "target_values_in_group": targets}
        out.update(row)
        target_conflict_rows.append(out)

duration_counter = Counter()
date_parse_errors = []
reg_dates = []
end_dates = []
for idx, row in enumerate(membership, start=2):
    try:
        reg = parse_date_yy(row["reg_date"])
        end = parse_date_yy(row["end_date"])
        reg_dates.append(reg)
        end_dates.append(end)
        duration_counter[(end - reg).days] += 1
    except ValueError as exc:
        date_parse_errors.append({"source": "Membership", "source_row_number": idx, "value": f"reg_date={row.get('reg_date','')}|end_date={row.get('end_date','')}", "error": str(exc)})
membership_duration_distribution = [{"duration_days": k, "row_count": v, "rate": round(v/len(membership), 6)} for k, v in sorted(duration_counter.items())]
membership_duration_distribution.append({"duration_days": "reg_date_range", "row_count": f"{min(reg_dates)} to {max(reg_dates)}" if reg_dates else "", "rate": ""})
membership_duration_distribution.append({"duration_days": "end_date_range", "row_count": f"{min(end_dates)} to {max(end_dates)}" if end_dates else "", "rate": ""})

ages = [safe_int(row.get("age", "")) for row in membership]
max_screens = [safe_int(row.get("max_screen", "")) for row in membership]
membership_value_anomaly_summary = []
def anomaly_row(section, metric, count, note):
    membership_value_anomaly_summary.append({"section": section, "metric": metric, "count": count, "rate": round(count/len(membership), 6), "note": note})
anomaly_row("max_screen", "missing_or_non_numeric", sum(1 for v in max_screens if v is None), "Audit only.")
valid_screen_values = {1, 2, 3, 4}
screen_counter = Counter(v for v in max_screens if v is not None)
anomaly_row("max_screen", "outside_1_to_4", sum(1 for v in max_screens if v is not None and v not in valid_screen_values), "Potential invalid/rare values; do not correct here.")
for value, count in sorted(screen_counter.items()):
    if count <= max(5, len(membership) * 0.001) or value not in valid_screen_values:
        anomaly_row("max_screen", f"rare_or_invalid_value_{value}", count, "Rare threshold: <=0.1% or <=5 rows, or outside expected 1-4.")
anomaly_row("age", "missing_or_non_numeric", sum(1 for v in ages if v is None), "Audit only.")
anomaly_row("age", "obvious_invalid_age_lt_0", sum(1 for v in ages if v is not None and v < 0), "Obvious invalid candidate.")
anomaly_row("age", "obvious_invalid_age_gt_100", sum(1 for v in ages if v is not None and v > 100), "Obvious invalid candidate.")
anomaly_row("gender", "missing", sum(1 for row in membership if row.get("gender", "") == ""), "Audit only.")
anomaly_row("gender", "not_M_or_F", sum(1 for row in membership if row.get("gender", "") not in {"M", "F", ""}), "Unexpected category candidate.")
anomaly_row("is_user_verified", "missing", sum(1 for row in membership if row.get("is_user_verified", "") == ""), "Audit only.")
anomaly_row("is_user_verified", "not_Y_or_N", sum(1 for row in membership if row.get("is_user_verified", "") not in {"Y", "N", ""}), "Unexpected category candidate.")
combo_counter = Counter((row.get("gender", ""), row.get("is_user_verified", ""), row.get("age", "")) for row in membership)
suspicious_combo_count = sum(c for (g, verified, age), c in combo_counter.items() if g == "" or verified == "" or age == "")
anomaly_row("gender_is_user_verified_age", "any_missing_in_combo", suspicious_combo_count, "Suspicious combination candidate: at least one of gender/is_user_verified/age is missing.")
for (g, verified, age), count in combo_counter.most_common(20):
    if g == "" or verified == "" or age == "":
        anomaly_row("gender_is_user_verified_age", f"combo_gender={g}|verified={verified}|age={age}", count, "Top suspicious missing combination.")

# 3. UserMapping audit
map_key_to_nums = defaultdict(list)
map_num_to_keys = defaultdict(list)
for row in mapping:
    map_key_to_nums[row["USER_KEY"]].append(row["USER_NUM"])
    map_num_to_keys[row["USER_NUM"]].append(row["USER_KEY"])
membership_user_keys = {row["USER_KEY"] for row in membership}
membership_rows_with_multi_map = sum(1 for row in membership if len(set(map_key_to_nums.get(row["USER_KEY"], []))) > 1)
usermapping_cardinality_audit = [
    {"section": "summary", "metric": "row_count", "value": "all", "count": len(mapping), "recommendation": "Audit only."},
    {"section": "summary", "metric": "USER_KEY_cardinality", "value": "unique_USER_KEY", "count": len(map_key_to_nums), "recommendation": "USER_KEY is join key to Membership."},
    {"section": "summary", "metric": "USER_NUM_cardinality", "value": "unique_USER_NUM", "count": len(map_num_to_keys), "recommendation": "USER_NUM is join key to View_History."},
    {"section": "pattern", "metric": "one_to_one_USER_KEY_to_USER_NUM", "value": "distinct_USER_NUM_count_eq_1", "count": sum(1 for nums in map_key_to_nums.values() if len(set(nums)) == 1), "recommendation": "Safe to map without expansion at USER_KEY level."},
    {"section": "pattern", "metric": "one_to_many_USER_KEY_to_USER_NUM", "value": "distinct_USER_NUM_count_gt_1", "count": sum(1 for nums in map_key_to_nums.values() if len(set(nums)) > 1), "recommendation": "Do not multiply Membership rows silently; aggregate logs back to membership_row_id."},
    {"section": "pattern", "metric": "many_to_one_USER_NUM_to_USER_KEY", "value": "distinct_USER_KEY_count_gt_1", "count": sum(1 for keys in map_num_to_keys.values() if len(set(keys)) > 1), "recommendation": "Review before grouping by USER_NUM."},
    {"section": "join_risk", "metric": "membership_rows_that_would_duplicate_on_join", "value": "rows", "count": membership_rows_with_multi_map, "recommendation": "Preserve membership_row_id and aggregate all mapped USER_NUM logs back to one row."},
]
for user_key, nums in map_key_to_nums.items():
    distinct = sorted(set(nums), key=lambda x: safe_int(x) if safe_int(x) is not None else x)
    if len(distinct) > 1:
        usermapping_cardinality_audit.append({"section": "one_to_many_detail", "metric": "USER_KEY_maps_to_multiple_USER_NUM", "value": user_key, "count": len(distinct), "recommendation": "|".join(distinct)})

# 4. ViewHistory audit
watch_times = [safe_float(row.get("watch_time(min)", "")) for row in views]
watch_times_clean = sorted(v for v in watch_times if v is not None)
watch_dates = []
for idx, row in enumerate(views, start=2):
    try:
        watch_dates.append(parse_date_yyyymmdd(row["watch_day"]))
    except ValueError as exc:
        date_parse_errors.append({"source": "View_History", "source_row_number": idx, "value": row.get("watch_day", ""), "error": str(exc)})
viewhistory_basic_audit = [
    {"section": "summary", "metric": "row_count", "value": "all", "count": len(views), "note": "Audit only."},
    {"section": "summary", "metric": "USER_NUM_cardinality", "value": "unique_USER_NUM", "count": len({row["USER_NUM"] for row in views}), "note": "Identifier only."},
    {"section": "summary", "metric": "MOVIE_NUM_cardinality", "value": "unique_MOVIE_NUM", "count": len({row["MOVIE_NUM"] for row in views}), "note": "Identifier only."},
    {"section": "watch_day", "metric": "watch_day_range", "value": f"{min(watch_dates)} to {max(watch_dates)}" if watch_dates else "", "count": len(watch_dates), "note": "Parsed from YYYYMMDD."},
    {"section": "watch_time", "metric": "min", "value": min(watch_times_clean) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "max", "value": max(watch_times_clean) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "mean", "value": round(mean(watch_times_clean), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "median", "value": median(watch_times_clean) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "p01", "value": round(percentile(watch_times_clean, 0.01), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "p05", "value": round(percentile(watch_times_clean, 0.05), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "p25", "value": round(percentile(watch_times_clean, 0.25), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "p75", "value": round(percentile(watch_times_clean, 0.75), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "p95", "value": round(percentile(watch_times_clean, 0.95), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "p99", "value": round(percentile(watch_times_clean, 0.99), 6) if watch_times_clean else "", "count": "", "note": "watch_time(min)."},
    {"section": "watch_time", "metric": "watch_time_le_0_count", "value": "<=0", "count": sum(1 for v in watch_times_clean if v <= 0), "note": "Do not delete here."},
    {"section": "watch_time", "metric": "watch_time_eq_1_count", "value": "1", "count": sum(1 for v in watch_times_clean if v == 1), "note": "Very short watch candidate."},
    {"section": "watch_time", "metric": "watch_time_le_5_count", "value": "<=5", "count": sum(1 for v in watch_times_clean if v <= 5), "note": "Very short watch candidate."},
]
view_cols = loaded["View_History"]["columns"]
view_full_counts = Counter(tuple(row.get(c, "") for c in view_cols) for row in views)
view_seq_key_counts = Counter((row["USER_NUM"], row["MOVIE_NUM"], row["watch_day"], row["watch_seq"]) for row in views)
view_same_movie_day_counts = Counter((row["USER_NUM"], row["MOVIE_NUM"], row["watch_day"]) for row in views)
viewhistory_duplicate_audit = [
    {"audit_item": "exact_duplicate_view_rows", "definition": "all View_History columns identical", "group_count": sum(1 for c in view_full_counts.values() if c > 1), "row_count": sum(c for c in view_full_counts.values() if c > 1), "extra_row_count": sum(c-1 for c in view_full_counts.values() if c > 1), "recommendation": "Classify and count only."},
    {"audit_item": "duplicate_like_by_user_movie_day_seq", "definition": "USER_NUM, MOVIE_NUM, watch_day, watch_seq identical", "group_count": sum(1 for c in view_seq_key_counts.values() if c > 1), "row_count": sum(c for c in view_seq_key_counts.values() if c > 1), "extra_row_count": sum(c-1 for c in view_seq_key_counts.values() if c > 1), "recommendation": "Classify and count only."},
    {"audit_item": "repeated_same_user_same_movie_same_day_logs", "definition": "USER_NUM, MOVIE_NUM, watch_day identical", "group_count": sum(1 for c in view_same_movie_day_counts.values() if c > 1), "row_count": sum(c for c in view_same_movie_day_counts.values() if c > 1), "extra_row_count": sum(c-1 for c in view_same_movie_day_counts.values() if c > 1), "recommendation": "May represent multiple sessions; do not delete without policy."},
]

# 5. Membership x ViewHistory temporal audit, audit-only join
key_to_nums = defaultdict(set)
for row in mapping:
    key_to_nums[row["USER_KEY"]].add(row["USER_NUM"])
views_by_num = defaultdict(list)
for row in views:
    views_by_num[row["USER_NUM"]].append(row)
temporal_counts = defaultdict(Counter)
temporal_total = Counter()
for membership_row_id, row in enumerate(membership, start=1):
    try:
        reg = parse_date_yy(row["reg_date"])
        end = parse_date_yy(row["end_date"])
        duration = (end - reg).days
    except ValueError:
        duration = "date_parse_error"
        continue
    mapped_nums = key_to_nums.get(row["USER_KEY"], set())
    if not mapped_nums:
        temporal_counts[duration]["membership_rows_without_mapping"] += 1
        temporal_total["membership_rows_without_mapping"] += 1
        continue
    matched = 0
    for user_num in mapped_nums:
        for view in views_by_num.get(user_num, []):
            matched += 1
            try:
                watch = parse_date_yyyymmdd(view["watch_day"])
            except ValueError:
                temporal_counts[duration]["watch_date_parse_error"] += 1
                temporal_total["watch_date_parse_error"] += 1
                continue
            temporal_counts[duration]["total_view_log_count"] += 1
            temporal_total["total_view_log_count"] += 1
            if watch < reg:
                temporal_counts[duration]["watch_date_lt_reg_date"] += 1
                temporal_total["watch_date_lt_reg_date"] += 1
            if watch == reg:
                temporal_counts[duration]["watch_date_eq_reg_date"] += 1
                temporal_total["watch_date_eq_reg_date"] += 1
            if reg <= watch <= end:
                temporal_counts[duration]["reg_date_le_watch_date_le_end_date"] += 1
                temporal_total["reg_date_le_watch_date_le_end_date"] += 1
            if watch == end:
                temporal_counts[duration]["watch_date_eq_end_date"] += 1
                temporal_total["watch_date_eq_end_date"] += 1
            if watch > end:
                temporal_counts[duration]["watch_date_gt_end_date"] += 1
                temporal_total["watch_date_gt_end_date"] += 1
    if matched == 0:
        temporal_counts[duration]["membership_rows_without_view_logs"] += 1
        temporal_total["membership_rows_without_view_logs"] += 1
temporal_fields = ["total_view_log_count", "watch_date_lt_reg_date", "watch_date_eq_reg_date", "reg_date_le_watch_date_le_end_date", "watch_date_eq_end_date", "watch_date_gt_end_date", "membership_rows_without_mapping", "membership_rows_without_view_logs", "watch_date_parse_error"]
membership_view_temporal_audit = []
for duration, counts in sorted(temporal_counts.items(), key=lambda x: str(x[0])):
    out = {"duration_days": duration, "end_date_inclusiveness_status": "unresolved_audit_only"}
    for field in temporal_fields:
        out[field] = counts.get(field, 0)
    out["total_count"] = counts.get("total_view_log_count", 0) + counts.get("membership_rows_without_mapping", 0) + counts.get("membership_rows_without_view_logs", 0) + counts.get("watch_date_parse_error", 0)
    membership_view_temporal_audit.append(out)

# 6. MovieMaster audit
movie_cols = loaded["Movie_Master"]["columns"]
movie_by_id = defaultdict(list)
for row_number, row in enumerate(movies, start=2):
    movie_by_id[row["MOVIE_NUM"]].append((row_number, row))
movie_id_view_counts = Counter(row["MOVIE_NUM"] for row in views)
moviemaster_duplicate_audit = [
    {"section": "summary", "MOVIE_NUM": "all", "source_row_number": "", "duplicate_group_size": "", "movie_master_row_count": len(movies), "movie_id_cardinality": len(movie_by_id), "duplicate_movie_id_count": sum(1 for g in movie_by_id.values() if len(g) > 1), "conflicting_columns": "", "expected_join_extra_rows_if_raw_joined_to_viewhistory": sum(movie_id_view_counts[mid] * (len(group)-1) for mid, group in movie_by_id.items() if len(group) > 1), "recommendation": "Deduplicate Movie_Master by MOVIE_NUM before joining to View_History; do not apply in Stage 01."}
]
for movie_id, group in movie_by_id.items():
    if len(group) <= 1:
        continue
    conflict_cols = [col for col in movie_cols if len({row.get(col, "") for _, row in group}) > 1]
    expected_extra = movie_id_view_counts[movie_id] * (len(group) - 1)
    for row_number, row in group:
        moviemaster_duplicate_audit.append({
            "section": "duplicate_movie_id_row",
            "MOVIE_NUM": movie_id,
            "source_row_number": row_number,
            "duplicate_group_size": len(group),
            "movie_master_row_count": "",
            "movie_id_cardinality": "",
            "duplicate_movie_id_count": "",
            "conflicting_columns": "|".join(conflict_cols),
            "expected_join_extra_rows_if_raw_joined_to_viewhistory": expected_extra,
            "recommendation": f"movie_title={row.get('movie_title','')}|ott_release_month={row.get('ott_release_month','')}|genre={row.get('genre','')}",
        })

# Write required CSV outputs
write_csv(TABLE_DIR / "01_v2_raw_file_inventory.csv", raw_file_inventory, ["dataset", "file_name", "relative_path", "sheet_name_if_applicable", "row_count", "column_count", "column_names", "inferred_dtypes", "missing_value_counts", "total_missing_values", "file_size_bytes"])
write_csv(TABLE_DIR / "01_v2_schema_summary.csv", schema_summary, ["dataset", "column_name", "inferred_dtype", "row_count", "missing_count", "missing_rate", "unique_count", "unique_count_excluding_missing", "min_value", "max_value", "top_values"])
write_csv(TABLE_DIR / "01_v2_membership_target_summary.csv", membership_target_summary, ["section", "metric", "value", "count", "rate", "note"])
write_csv(TABLE_DIR / "01_v2_membership_duplicate_audit.csv", membership_duplicate_audit, ["audit_item", "definition", "group_count", "row_count", "extra_row_count", "recommendation"])
write_csv(TABLE_DIR / "01_v2_membership_target_conflict_rows.csv", target_conflict_rows, ["conflict_level", "conflict_id", "source_row_number", "group_size", "target_values_in_group"] + membership_cols)
write_csv(TABLE_DIR / "01_v2_membership_duration_distribution.csv", membership_duration_distribution, ["duration_days", "row_count", "rate"])
write_csv(TABLE_DIR / "01_v2_membership_value_anomaly_summary.csv", membership_value_anomaly_summary, ["section", "metric", "count", "rate", "note"])
write_csv(TABLE_DIR / "01_v2_usermapping_cardinality_audit.csv", usermapping_cardinality_audit, ["section", "metric", "value", "count", "recommendation"])
write_csv(TABLE_DIR / "01_v2_viewhistory_basic_audit.csv", viewhistory_basic_audit, ["section", "metric", "value", "count", "note"])
write_csv(TABLE_DIR / "01_v2_viewhistory_duplicate_audit.csv", viewhistory_duplicate_audit, ["audit_item", "definition", "group_count", "row_count", "extra_row_count", "recommendation"])
write_csv(TABLE_DIR / "01_v2_membership_view_temporal_audit.csv", membership_view_temporal_audit, ["duration_days", "end_date_inclusiveness_status"] + temporal_fields + ["total_count"])
write_csv(TABLE_DIR / "01_v2_moviemaster_duplicate_audit.csv", moviemaster_duplicate_audit, ["section", "MOVIE_NUM", "source_row_number", "duplicate_group_size", "movie_master_row_count", "movie_id_cardinality", "duplicate_movie_id_count", "conflicting_columns", "expected_join_extra_rows_if_raw_joined_to_viewhistory", "recommendation"])

raw_after = {name: {"mtime_ns": path.stat().st_mtime_ns, "size": path.stat().st_size} for name, path in RAW_FILES.items() if path.exists()}
required_paths = [TABLE_DIR / name for name in REQUIRED_CSVS if name != "01_v2_audit_final_checks.csv"]
report_path = DATA_DIR / MARKDOWN_REPORT
raw_files_found = all(path.exists() for path in RAW_FILES.values())
raw_files_unmodified = raw_before == raw_after

confirmed_facts = [
    f"Active v2 raw files inspected: {', '.join(RAW_FILES.keys())}.",
    f"Membership rows: {len(membership)}; target distribution: {dict(target_counts)}.",
    f"Membership unique USER_KEY count: {len(user_key_counts)}; duplicate USER_KEY keys: {sum(1 for c in user_key_counts.values() if c > 1)}.",
    f"Strict target-conflict groups: {len(strict_conflict_groups)}; strict conflict rows: {sum(len(g) for g in strict_conflict_groups.values())}.",
    f"User_Mapping one-to-many USER_KEY count: {sum(1 for nums in map_key_to_nums.values() if len(set(nums)) > 1)}.",
    f"View_History rows: {len(views)}; watch_day range: {min(watch_dates)} to {max(watch_dates)}.",
    f"Movie_Master duplicate MOVIE_NUM count: {sum(1 for g in movie_by_id.values() if len(g) > 1)}.",
]
suspicious_issues = [
    "Strict target-conflict rows exist and require label-policy review before modeling.",
    "Some USER_KEY values map to multiple USER_NUM values; joins must not multiply Membership rows silently.",
    "Movie_Master has duplicate MOVIE_NUM rows; raw join to View_History would create multiplication.",
    "watch_date == end_date exists and end_date inclusiveness remains unresolved.",
    "Very short watch logs exist and should be classified before deciding whether to keep, downweight, or exclude them.",
]
unresolved_questions = [
    "Is end_date inclusive or exclusive for behavior observation and leakage control?",
    "Should strict target-conflict rows be excluded, flagged, or resolved by a documented business rule?",
    "Should one USER_KEY mapping to multiple USER_NUM values be interpreted as multiple devices/accounts or mapping duplication?",
    "Which Movie_Master duplicate row should represent a MOVIE_NUM when metadata differs?",
    "Are 1-minute and <=5-minute watch logs meaningful engagement, accidental playback, or noise?",
]
recommended_candidates = [
    "Assign membership_row_id before any joins and preserve one final row per membership_row_id.",
    "Aggregate all mapped USER_NUM logs back to membership_row_id instead of expanding Membership rows.",
    "Deduplicate Movie_Master by MOVIE_NUM before any content-feature join, with conflict audit retained.",
    "Keep w1_3 and w1_4 feature windows separate in later stages.",
    "Exclude raw identifiers and raw dates from model features in later stages.",
]
v1_plausible = [
    "The broad workflow EDA -> modeling -> interpretation -> segmentation -> retention strategy remains plausible as a business workflow.",
    "Leakage-controlled baseline modeling remains a necessary next step.",
    "Audit artifacts should be retained for every exclusion or correction decision.",
]
v1_revalidate = [
    "All old row counts and AUC values must be recomputed from v2.",
    "Old 1-3 week observation-window assumptions must be revalidated against v2, especially with week-4 behavior available.",
    "Old preprocessing rules for duplicates, short watches, end_date handling, and Movie_Master metadata must be revalidated.",
    "Old feature conclusions must not be carried forward without v2 evidence.",
]
output_files = [rel(TABLE_DIR / name) for name in REQUIRED_CSVS] + [rel(report_path), rel(DATA_DIR / "01_v2_audit_summary.json")]

report_lines = []
report_lines.append("# 01_v2 Data Audit Report")
report_lines.append("")
report_lines.append("## What Was Inspected")
for item in [rel(path) for path in RAW_FILES.values()]:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## What Was Not Inspected")
for item in ["__legacy", "park.ingyeom/__legacy", "old reports", "old handoff documents", "team member folders", "preprocessed/interim datasets"]:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Confirmed Facts")
for item in confirmed_facts:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Suspicious Issues")
for item in suspicious_issues:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Unresolved Business-Definition Questions")
for item in unresolved_questions:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Recommended Preprocessing Candidates")
for item in recommended_candidates:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Old v1 Assumptions Still Plausible")
for item in v1_plausible:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Old v1 Assumptions That Must Be Revalidated")
for item in v1_revalidate:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Exact Output File List")
for item in output_files:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Final Checks")
report_lines.append("See `01_v2_audit_final_checks.csv` for pass/fail validation.")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

write_json(DATA_DIR / "01_v2_audit_summary.json", {
    "raw_row_counts": {name: len(info["rows"]) for name, info in loaded.items()},
    "target_distribution": dict(target_counts),
    "strict_target_conflict_groups": len(strict_conflict_groups),
    "strict_target_conflict_rows": sum(len(g) for g in strict_conflict_groups.values()),
    "core_event_conflict_groups": len(core_conflict_groups),
    "movie_master_duplicate_movie_id_count": sum(1 for g in movie_by_id.values() if len(g) > 1),
    "expected_moviemaster_join_extra_rows": moviemaster_duplicate_audit[0]["expected_join_extra_rows_if_raw_joined_to_viewhistory"],
    "temporal_total": dict(temporal_total),
    "output_files": output_files,
})

required_csvs_created = all((TABLE_DIR / name).exists() for name in REQUIRED_CSVS if name != "01_v2_audit_final_checks.csv")
final_checks = [
    {"check": "raw_files_found", "status": "PASS" if raw_files_found else "FAIL", "detail": "All expected files under _data/01_raw found."},
    {"check": "no_raw_files_modified", "status": "PASS" if raw_files_unmodified else "FAIL", "detail": "Raw file size and mtime unchanged during notebook execution."},
    {"check": "all_required_csvs_created", "status": "PASS" if required_csvs_created else "FAIL", "detail": "All required CSV outputs except final_checks exist before final_checks write."},
    {"check": "target_conflict_audit_created", "status": "PASS" if (TABLE_DIR / "01_v2_membership_target_conflict_rows.csv").exists() else "FAIL", "detail": f"strict_conflict_groups={len(strict_conflict_groups)}"},
    {"check": "duration_distribution_created", "status": "PASS" if (TABLE_DIR / "01_v2_membership_duration_distribution.csv").exists() else "FAIL", "detail": f"duration_groups={len(duration_counter)}"},
    {"check": "temporal_audit_created", "status": "PASS" if (TABLE_DIR / "01_v2_membership_view_temporal_audit.csv").exists() else "FAIL", "detail": "end_date inclusiveness labeled unresolved."},
    {"check": "MovieMaster_duplicate_audit_created", "status": "PASS" if (TABLE_DIR / "01_v2_moviemaster_duplicate_audit.csv").exists() else "FAIL", "detail": f"duplicate_MOVIE_NUM_count={sum(1 for g in movie_by_id.values() if len(g) > 1)}"},
    {"check": "markdown_report_created", "status": "PASS" if report_path.exists() else "FAIL", "detail": rel(report_path)},
]
write_csv(TABLE_DIR / "01_v2_audit_final_checks.csv", final_checks, ["check", "status", "detail"])

print("01_v2 data overview and anomaly audit completed.")
print("Final checks:")
for row in final_checks:
    print(f"{row['check']}: {row['status']} - {row['detail']}")


01_v2 data overview and anomaly audit completed.
Final checks:
raw_files_found: PASS - All expected files under _data/01_raw found.
no_raw_files_modified: PASS - Raw file size and mtime unchanged during notebook execution.
all_required_csvs_created: PASS - All required CSV outputs except final_checks exist before final_checks write.
target_conflict_audit_created: PASS - strict_conflict_groups=35
duration_distribution_created: PASS - duration_groups=21
temporal_audit_created: PASS - end_date inclusiveness labeled unresolved.
MovieMaster_duplicate_audit_created: PASS - duplicate_MOVIE_NUM_count=380
markdown_report_created: PASS - park.ingyeom/reports/data/01_v2_data_overview_and_audit/01_v2_data_audit_report.md
